# Segment Creator and Analyser

In [ ]:
import torch
import torchvision
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())
import sys
!{sys.executable} -m pip install opencv-python matplotlib
!{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git'

!mkdir images
!wget -P images https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/dog.jpg

!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [ ]:
import matplotlib.pyplot as plt
import cv2
from mpl_toolkits.axes_grid1 import ImageGrid
import numpy as np

In [ ]:
import sys
sys.path.append("..")
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"

device = "cuda"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(sam)

In [ ]:
def get_mask(input_path):

  image = cv2.imread(input_path)
  image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  plt.imshow(image)
  plt.show()

  plt.figure()

  masks = mask_generator.generate(image)

  fig = plt.figure(figsize=(20., 20.))
  grid = ImageGrid(fig, 111,
                  nrows_ncols=(int((len(masks)+9)/10), 10),
                  axes_pad=0.1,
                  )

  l = []
  for i in range(len(masks)):
    l.append(masks[i]['segmentation'])

  for ax, im in zip(grid, l):
      ax.imshow(im)

  return masks

In [ ]:
def compute_core_spurious(mask_nb, heatmap_paths, masks):
    for i, heatmap in enumerate(heatmap_paths):
      K = torch.tensor(masks[mask_nb]['segmentation'])
      img = cv2.imread(heatmap, cv2.IMREAD_GRAYSCALE)
      img = (img - img.min()) / (img.max() - img.min())
      core_mean = img[K].mean()
      K = ~K
      spurious_mean = img[K].mean()
      print('Core %     description_{}:'.format(i), core_mean/(core_mean+spurious_mean))
      print('Spurious % description_{}:'.format(i), spurious_mean/(core_mean+spurious_mean))

## Change BELOW!!

In [ ]:
input_path = './results/golden_retriever/golden_retriever_input.jpg'
save_masks = get_mask(input_path)

In [ ]:
mask_nb = 0
total_gradcams = 2
list_of_gradcams = []
for i in range(total_gradcams):
  list_of_gradcams.append('/'.join(input_path.split('/')[:-1]) + '/' + 'g_' + '_'.join(input_path.split('/')[-1].split('_')[:-1]) + '_gradcam_desc_{}.jpg'.format(i))
compute_core_spurious(mask_nb, list_of_gradcams, save_masks)